## Fidelity of MPDO from sequential circuit

In [1]:
include("evolMPDO.jl")

MPDO_to_dense (generic function with 1 method)

In [2]:
include("IsingED.jl")

compute_fidelity (generic function with 1 method)

## Ising GS + X dephasing - exact fidelity

In [3]:
N = 8;
h = 1.2; # slightly off critical
Es, psis = get_Ising_eigenstates(N,h);
@show Es[1]
psi_gs = psis[:,1];

Es[1] = -11.392199651525441


In [4]:
p1 = 0.2;
p2 = 0.4;
rho1 = apply_channel!(psi_gs*psi_gs',dephasing_X(p1),N)
rho2 = apply_channel!(psi_gs*psi_gs',dephasing_X(p2),N);

In [5]:
compute_fidelity(rho1,rho2) # ||sqrt(rho1)*sqrt(rho2)||_1

0.9400387194933517

In [6]:
tr(rho1*rho2)/sqrt(tr(rho1^2)*tr(rho2^2)) # overlap in Hilbert-Schmidt norm

0.9148317195375935

## Ising GS + X dephasing - MPDO optimization

In [7]:
psiMPS = Ising_GS_DMRG(N,h);

After sweep 1 energy=-11.39133723900215  maxlinkdim=16 maxerr=1.01E-06 time=26.250
After sweep 2 energy=-11.392199554037292  maxlinkdim=15 maxerr=1.11E-07 time=0.010
After sweep 3 energy=-11.392199651092078  maxlinkdim=13 maxerr=1.11E-08 time=0.021
After sweep 4 energy=-11.39219965110114  maxlinkdim=13 maxerr=1.11E-08 time=0.009
After sweep 5 energy=-11.392199651101134  maxlinkdim=13 maxerr=1.11E-08 time=0.009
After sweep 6 energy=-11.392199651101139  maxlinkdim=13 maxerr=2.46E-11 time=0.294
After sweep 7 energy=-11.392199651101155  maxlinkdim=13 maxerr=2.46E-11 time=0.012
After sweep 8 energy=-11.39219965110114  maxlinkdim=13 maxerr=2.46E-11 time=0.012
After sweep 9 energy=-11.392199651101137  maxlinkdim=13 maxerr=2.46E-11 time=0.022
After sweep 10 energy=-11.392199651101132  maxlinkdim=13 maxerr=2.46E-11 time=0.008
After sweep 11 energy=-11.392199651101137  maxlinkdim=13 maxerr=2.46E-11 time=0.008
After sweep 12 energy=-11.392199651101155  maxlinkdim=13 maxerr=2.46E-11 time=0.008
Aft

In [8]:
M = myMPS(MPS_to_array(psiMPS));

In [9]:
Ws1 = Array{Float64,3}[purified_dephasing_channel(p1,[1,0,0]) for _ in 1:N]
M1 = add_noise_MPS(M, Ws1)

Ws2 = Array{Float64,3}[purified_dephasing_channel(p2,[1,0,0]) for _ in 1:N]
M2 = add_noise_MPS(M, Ws2);

In [10]:
function MPS_to_dense(M::myMPS{T}) where T
    ## Only run this for small systems!!
    d = phys_dim(M)
    L = length(M)
    tmp = M.TensorList[1]
    for i in 2:L
        A = M.TensorList[i]
        dR = size(A,3)
        @tensor tmp2[l,s1,s2,r] := tmp[l,s1,r1]*A[r1,s2,r]
        tmp = reshape(tmp2,(1,d^i,dR))
    end
    psi = reshape(tmp,d^L)
    #return psi
end

function MPDO_to_MPS(M::myMPDO{T}) where T
    ## MPS in doubled space
    Ts = Array{T,3}[]
    for i in 1:length(M)
        DL,dS,dA,DR = size(M.TensorList[i])
        push!(Ts,reshape(M.TensorList[i],(DL,dS*dA,DR)))
    end
    return myMPS(Ts)
end

MPDO_to_MPS (generic function with 1 method)

In [11]:
function MPDO_to_dense(M::myMPDO{T}) where T
    dS = phys_dim(M)
    dA = ancilla_dim(M)
    L = length(M)
    tmp = M.TensorList[1]
    for i in 2:L
        A = M.TensorList[i]
        dR = size(A,4)
        @tensor tmp2[l,s1,s2,a1,a2,r] := tmp[l,s1,a1,r1]*A[r1,s2,a2,r]
        tmp = reshape(tmp2,(1,dS^i,dA^i,dR))
    end
    rho = reshape(tmp,(dS^L,dA^L))
    return rho
end

MPDO_to_dense (generic function with 1 method)

In [12]:
## Test for small sizes
let
    M1dense = MPDO_to_dense(M1);
    M2dense = MPDO_to_dense(M2);

    rho1dense = M1dense*M1dense'
    rho2dense = M2dense*M2dense'
    compute_fidelity(rho1dense,rho2dense) 
end

0.9400387194342602

In [13]:
compute_overlap(M1,M2)  ## fidelity of purification without optimization

0.9223681600000002

In [14]:
M1cp,M2cp,ov = optimize_overlap(M1,M2,5); ## seems stuck!!!

----iteration 1 -----
Bond dimensions: 23,18
Overlap: 0.9325164889990571
----iteration 2 -----
Bond dimensions: 23,18
Overlap: 0.9325256032444098
----iteration 3 -----
Bond dimensions: 23,18
Overlap: 0.9325256120886778
----iteration 4 -----
Bond dimensions: 23,18
Overlap: 0.9325256120902865
----iteration 5 -----
Bond dimensions: 23,18
Overlap: 0.9325256120901646


In [15]:
random_unitary_layer_ancilla_onsite!(M2);

In [16]:
compute_overlap(M1,M2)  ## fidelity of purification after random reshuffle

4.179669745805817e-5

In [17]:
M1cp,M2cp,ov = optimize_overlap(M1,M2,5); ## still stuck!!!

----iteration 1 -----
Bond dimensions: 26,26
Overlap: 0.9198179339520343
----iteration 2 -----
Bond dimensions: 30,26
Overlap: 0.9215994598922763
----iteration 3 -----
Bond dimensions: 29,26
Overlap: 0.921601499411307
----iteration 4 -----
Bond dimensions: 29,26
Overlap: 0.92160151278693
----iteration 5 -----
Bond dimensions: 29,26
Overlap: 0.9216015129408323
